<div align="center">

## 基于 LSTM 的古诗生成

</div>

<div align="center">微电子学院 &nbsp;&nbsp; BC25219017 &nbsp;&nbsp; 宋城啸</div>

<div align="center">2026.5.10</div>

**课程：** 神经网络及其应用  
**任务：** 以「明月」为起始词，基于 LSTM 生成七言绝句

---

#### 摘要

古诗自动生成要求模型在高度受限的文本形式中维持语义连贯性，七言绝句严苛的格律约束使该任务尤具挑战。本文基于宋诗数据集，从 255 个 JSON 文件中筛选出 23,213 首七言绝句，构建字符级词表（|V|=6,564），训练了一个 3 层 LSTM 语言模型（hidden size=1024, embedding dim=512）。模型引入权重共享机制，将输出层与嵌入层绑定，参数量由 33.18M 降至 26.98M；训练使用 Dropout=0.3 正则化，并采用 StepLR（γ=0.7, step=15）控制学习率衰减。经 60 轮训练，训练损失由 6.57 降至 2.64，对应困惑度由 1110 降至 14.06，较初始配置（γ=0.5, step=10）改善明显。生成阶段以「明月」为起始词，配合温度采样（T=0.8）与强制标点约束，模型可稳定输出格律正确的七言绝句，但在起始位置重复率和跨句主题统一性方面仍存在问题。

**关键词**：LSTM；古诗生成；字符级语言模型；权重共享；困惑度

### 1. 研究背景与任务说明

中国古典诗词中，七言绝句以固定的篇幅（四句、每句七字）和精炼的表意方式著称，是检验序列模型在受限文本上建模能力的合适任务。近年来，基于循环神经网络及其变体的文本生成方法得到了广泛研究，将此类模型应用于古诗创作也积累了一定工作[3,4]。

本文选取「明月」为起始词，目标是训练一个字符级语言模型，使其能按七言绝句格律自动续写完整的四句诗。具体而言，本文关注三个问题：（1）字符级 LSTM 在小型古诗数据集上的训练稳定性与收敛行为；（2）权重共享与学习率衰减策略对模型性能的影响；（3）生成结果的质量评估与主要失败模式分析。

本文后续组织如下：第 2 节回顾 LSTM 与字符级语言模型的基础理论；第 3 节描述数据处理与词表构建过程；第 4 节给出模型架构设计及设计依据；第 5 节报告训练配置与结果；第 6 节展示生成样例并分析质量问题；第 7 节总结全文并讨论后续方向。

### 2. 理论基础

#### 2.1 循环神经网络与长短期记忆网络

前馈网络处理序列数据时，各输入被独立对待，无法利用元素间的先后关系。循环神经网络（RNN）在隐藏层引入循环连接，将上一时刻的隐藏状态馈入当前时刻，从而在计算图中纳入时间维度。然而，标准 RNN 在反向传播中存在梯度消失问题：序列增长时梯度沿时间方向指数级衰减，模型难以捕捉长距离依赖。

LSTM（Hochreiter & Schmidhuber, 1997）[1] 针对上述问题引入了门控机制与细胞状态。细胞状态 $C_t$ 沿时间轴传递，梯度可在其上较为顺畅地流动，为长距离建模提供了结构保障。LSTM 单元包含三个门：

| 门 | 公式 | 功能 |
|----|------|------|
| 遗忘门 | $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ | 控制旧信息的丢弃比例 |
| 输入门 | $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ | 控制新信息的写入比例 |
| 输出门 | $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ | 控制隐藏状态的输出比例 |

细胞状态按 $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$ 更新，隐藏状态为 $h_t = o_t \odot \tanh(C_t)$。多层 LSTM 堆叠可逐层抽取更抽象的序列特征，本文采用 3 层结构，具体考量见第 4 节。

#### 2.2 字符级语言模型

语言模型的核心是估计序列的联合概率分布：

$$P(w_1, \dots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_1, \dots, w_{t-1})$$

本文选择字符级建模而非词级建模，基于两点考量。其一，中文分词本身是开放问题，古汉语的分词尤其困难；其二，古诗中单字常直接承载完整语义，字符粒度更为契合。

训练阶段采用 Teacher Forcing——每个时间步的输入使用真实序列中的前一字符而非模型自身的上一步预测，这有利于加速收敛和稳定训练，但也引入了训练与推理时输入分布不一致的问题（Exposure Bias）。生成阶段切换为自回归采样：每次取上一步输出作为下一步输入，通过温度参数 $T$ 调控分布锐度——$T < 1$ 使分布更集中，$T > 1$ 使采样更多样。

#### 2.3 权重共享

权重共享（Press & Wolf, 2017）[2] 将输入嵌入矩阵 $\mathbf{E} \in \mathbb{R}^{|V| \times d}$ 与输出层权重绑定（$\mathbf{W} = \mathbf{E}^T$），核心收益有两方面：一是减少约一半嵌入相关参数；二是强制输入空间与输出空间共享语义结构，起到隐式正则化效果。该技术要求嵌入维度与输出层输入维度一致；当 LSTM 隐藏维度与嵌入维度不同时，需插入投影层完成维度对齐，本文第 4 节将给出具体设计。

### 3. 实验数据与预处理

#### 3.1 数据集说明

实验数据为宋代诗歌 JSON 数据集，共计 255 个 `poet.song.*.json` 文件。原始数据覆盖多种诗体，本文仅筛选七言绝句作为研究对象。筛选规则为：每首诗恰好包含两联，每联恰好 16 个字符（7 字 + 句中逗号 + 7 字 + 句末标点），以正则表达式 `^[一-鿿]{7}[，,][一-鿿]{7}[。！？]$` 逐联匹配。满足条件的样本共 23,213 首，每首展开为含标点的 32 字符序列，全部用于模型训练。

In [ ]:
import json, os, re, random
import numpy as np
import matplotlib
matplotlib.use('Agg')  # 服务器上没有 GUI，用 Agg 后端
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ─── 配置 ──────────────────────────────────────────────────────────
# 超参数都在这里改，方便跑实验对比
CONFIG = {
    # 数据：读前 N 个 JSON 文件，-1 表示全读
    "num_data_files": 80,
    "data_dir": "data_files",

    # 模型
    "embedding_dim": 512,
    "hidden_size":   1024,
    "num_layers":    3,
    "dropout":       0.3,    # 试过 0.1 过拟合太严重，0.5 又太欠拟合

    # 训练
    "batch_size":    128,
    "num_epochs":    60,
    "learning_rate": 1e-3,
    "lr_decay_step": 15,     # 之前用 10 衰减太快，后期 LR 接近 0 学不动
    "lr_decay_gamma":0.7,    # 之前用 0.5 太猛，0.7 温和一些
    "clip_grad":     5.0,    # 梯度裁剪，防爆炸
    "seed":          42,

    # 生成
    "start_words":   "明月",
    "temperature":   0.8,    # 0.8 偏保守，1.0 太随机容易出怪字

    # 输出
    "save_model":    "poem_lstm.pth",
    "loss_fig":      "training_loss.png",
}

# 固定随机种子，保证实验可复现
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# 特殊 token
PAD, START, END, UNK = "<PAD>", "<START>", "<END>", "<UNK>"

#### 3.2 序列格式与词表构建

每首七言绝句两联合并后形成长度为 32 的字符串，每个字符（含标点）视为一个 token。词表由两类符号构成：4 个特殊标记（`<PAD>`、`<START>`、`<END>`、`<UNK>`）和全部 6,560 个出现过的汉字/标点，共 6,564 个条目。字符按 Unicode 排序后依次编号，以保证不同运行间映射关系一致。

In [ ]:
# 七言绝句的正则：7个汉字 + 逗号 + 7个汉字 + 句末标点
_QIYAN_PATTERN = re.compile(
    r'^[一-鿿]{7}[，,][一-鿿]{7}[。！？]$'
)

def is_qiyan_jueju(paragraphs):
    """检查是否为七言绝句：必须恰好 2 联，每联 16 字"""
    if len(paragraphs) != 2:
        return False
    for p in paragraphs:
        ps = p.strip()
        if len(ps) != 16 or not _QIYAN_PATTERN.match(ps):
            return False
    return True


def load_sequences(data_dir, filenames):
    """从 JSON 文件里读取七言绝句，展开成 32 字符的序列"""
    sequences = []
    for fname in filenames:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f"[warning] 文件不存在，跳过: {fpath}")
            continue
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        for item in data:
            para = item.get("paragraphs", [])
            if is_qiyan_jueju(para):
                seq = "".join(p.strip() for p in para)  # 两联合并，长度 32
                sequences.append(seq)

    print(f"过滤后七言绝句: {len(sequences)} 首")
    assert len(sequences) > 0, "未找到七言绝句！请检查 data_dir 配置。"
    return sequences


def build_vocab(sequences):
    """统计所有出现过的字符，建字符→索引的映射"""
    chars = sorted(set("".join(sequences)))
    vocab = [PAD, START, END, UNK] + chars
    char2idx = {c: i for i, c in enumerate(vocab)}
    idx2char = {i: c for c, i in char2idx.items()}
    print(f"词表大小: {len(vocab)}")
    return char2idx, idx2char, vocab


# 扫描数据目录，按文件名排序取前 N 个
all_json = sorted([
    f for f in os.listdir(CONFIG["data_dir"])
    if f.startswith("poet.song") and f.endswith(".json")
])
n = CONFIG["num_data_files"]
data_files = all_json[:n] if n > 0 else all_json
print(f"共发现 {len(all_json)} 个 JSON 文件，将读取其中 {len(data_files)} 个")

sequences = load_sequences(CONFIG["data_dir"], data_files)
char2idx, idx2char, vocab = build_vocab(sequences)

# 看几条数据长什么样
print("\n示例序列：")
for s in sequences[:3]:
    print(" ", s)

#### 3.3 样本构造

采用 Teacher Forcing 方式构造训练样本。对每首 32 字符的诗，生成一组 `(inp, tgt)` 对：输入序列为 `[START] + seq[:31]`，即每个位置送入前一个字符；目标序列为 `seq[:32]`，即每个位置的标签为当前字符。模型在每个时间步的任务是：给定前文，预测下一字符。

In [ ]:
class PoemDataset(Dataset):
    """
    把诗句序列转成 (inp, tgt) 对：
      inp = [START] + seq[:-1]   每个位置输入"前一个字"
      tgt = seq                  每个位置标签是"当前字"
    """
    def __init__(self, sequences, char2idx):
        start_id = char2idx[START]
        unk_id   = char2idx[UNK]
        self.data = []
        for seq in sequences:
            ids = [char2idx.get(c, unk_id) for c in seq]
            inp = torch.tensor([start_id] + ids[:-1], dtype=torch.long)
            tgt = torch.tensor(ids, dtype=torch.long)
            self.data.append((inp, tgt))

    def __len__(self):            return len(self.data)
    def __getitem__(self, i):     return self.data[i]


dataset = PoemDataset(sequences, char2idx)
print(f"训练样本数: {len(dataset)}")
inp_sample, tgt_sample = dataset[0]
print(f"inp shape: {inp_sample.shape},  tgt shape: {tgt_sample.shape}")

### 4. 模型架构设计

#### 4.1 网络结构

模型按以下流程组织：字符经嵌入层映射为稠密向量后，送入 3 层堆叠 LSTM 进行序列建模，LSTM 输出经投影层降维，最后由与嵌入层共享权重的线性层产生词表维度的 logits。

```
输入 (B, 32)
  → Embedding(6564, 512, padding_idx=0)     字符嵌入
  → Dropout(0.3)
  → 3-Layer LSTM(512 → 1024, dropout=0.3)   序列建模
  → Dropout(0.3)
  → Linear(1024 → 512, bias=False)          维度投影
  → Linear(512 → 6564, bias=False)          输出（与 Embedding 权重共享）
  → 输出 logits (B, 32, 6564)
```

#### 4.2 设计决策

**LSTM 层数（3 层）。** 古诗篇幅虽短，但字面意象、情感层次和对仗关系需要多层抽象才能有效建模。对比实验表明，3 层配置在训练损失上显著优于 2 层；增至 4 层后参数量上升明显且出现过拟合倾向，故取 3 层。

**隐藏维度（1024）。** 古诗中上下文依赖关系较复杂——对仗、押韵、意象呼应均跨越多个字——较大的隐藏维度有利于编码更丰富的上下文信息。代价是参数量增大，需配合 Dropout 正则化。

**Dropout（0.3）。** 模型约 27M 参数量，训练样本仅约 74 万 token，参数/token 比约 36.3，过拟合风险突出。实验比较了 0.1、0.3、0.5 三个取值：0.1 时生成结果重复严重，0.5 时收敛过慢，0.3 在二者间取得较好平衡。

**权重共享。** 参照 Press & Wolf (2017)[2]，将输出层 `fc.weight` 绑定至 `embedding.weight`。因 LSTM hidden size (1024) 与 embedding dim (512) 不一致，中间加入无偏置线性投影层完成维度对齐。绑定后参数量由 33.18M 降至 26.98M，降幅约 19%。

In [ ]:
class PoemLSTM(nn.Module):
    """
    字符级 LSTM 语言模型。
    结构：Embedding → 3层LSTM → Dropout → 投影层 → 输出层(共享权重)
    
    权重共享的思路：把 fc.weight 和 embedding.weight 绑定，
    省参数的同时强制输入输出用同一套语义空间。
    因为 LSTM hidden_size(1024) ≠ emb_dim(512)，中间加了个投影层做维度对齐。
    """
    def __init__(self, vocab_size, emb_dim, hidden_size, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size  = emb_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        # 投影层：把 LSTM 输出从 1024 降到 512，才能和 embedding 共享
        self.proj = nn.Linear(hidden_size, emb_dim, bias=False)
        # 输出层不加 bias（因为要和 embedding 绑定，embedding 没有 bias）
        self.fc   = nn.Linear(emb_dim, vocab_size, bias=False)
        self.fc.weight = self.embedding.weight  # 权重绑定

        nn.init.xavier_uniform_(self.proj.weight)

    def forward(self, x, hidden=None):
        """输入 (B, L)，输出 logits (B, L, V) 和 hidden state"""
        emb    = self.dropout(self.embedding(x))        # (B, L, E)
        out, hidden = self.lstm(emb, hidden)             # (B, L, H)
        out    = self.proj(self.dropout(out))            # (B, L, E)
        logits = self.fc(out)                            # (B, L, V)
        return logits, hidden

    def init_hidden(self, batch, device):
        """初始化 h0 和 c0 为全零"""
        h = torch.zeros(self.lstm.num_layers, batch,
                        self.lstm.hidden_size, device=device)
        return (h, torch.zeros_like(h))


# 实例化
model = PoemLSTM(
    vocab_size  = len(vocab),
    emb_dim     = CONFIG["embedding_dim"],
    hidden_size = CONFIG["hidden_size"],
    num_layers  = CONFIG["num_layers"],
    dropout     = CONFIG["dropout"],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型参数量: {n_params:,}")
print(model)

### 5. 模型训练

#### 5.1 训练配置

优化器选用 Adam（β₁=0.9, β₂=0.999），初始学习率 1×10⁻³，配合 StepLR 调度器每 15 轮将学习率乘以 0.7。该配置基于前期实验调整得出：最初的方案中每 10 轮衰减 0.5，导致第 60 轮学习率降至 1.56×10⁻⁵，后期几乎无有效学习；调整为当前设置后，第 60 轮学习率为 2.40×10⁻⁴，训练在后期仍可推进。各阶段学习率如下：

| 阶段 | Epoch | 学习率 |
|------|-------|--------|
| 1 | 1–14 | 1.00×10⁻³ |
| 2 | 15–29 | 7.00×10⁻⁴ |
| 3 | 30–44 | 4.90×10⁻⁴ |
| 4 | 45–59 | 3.43×10⁻⁴ |
| 5 | 60 | 2.40×10⁻⁴ |

损失函数为 CrossEntropyLoss，逐 token 计算。梯度裁剪阈值设为 5.0。每 5 个 epoch 输出一次困惑度（PPL = exp(Loss)）以跟踪训练进展。

In [ ]:
import math

def train_epoch(model, loader, optimizer, criterion, device, clip):
    """跑一个 epoch，返回平均 per-token loss"""
    model.train()
    total_loss, total_n = 0.0, 0
    for inp, tgt in loader:
        inp, tgt = inp.to(device), tgt.to(device)
        hidden = model.init_hidden(inp.size(0), device)

        logits, _ = model(inp, hidden)
        # 把 (B, L, V) 和 (B, L) 展平成 2D 和 1D，方便算 CE loss
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)  # 防梯度爆炸
        optimizer.step()

        total_loss += loss.item() * tgt.numel()
        total_n    += tgt.numel()
    return total_loss / total_n


# ─── 训练组件 ──────────────────────────────────────────────────────
loader    = DataLoader(dataset, batch_size=CONFIG["batch_size"],
                       shuffle=True, num_workers=0)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=CONFIG["lr_decay_step"],
    gamma=CONFIG["lr_decay_gamma"])
criterion = nn.CrossEntropyLoss()

# ─── 开始训练 ──────────────────────────────────────────────────────
epoch_losses = []

for epoch in range(1, CONFIG["num_epochs"] + 1):
    loss = train_epoch(model, loader, optimizer, criterion,
                       DEVICE, CONFIG["clip_grad"])
    scheduler.step()
    epoch_losses.append(loss)

    lr_now = optimizer.param_groups[0]["lr"]

    # 每 5 轮打一次 PPL，方便观察训练进展
    if epoch % 5 == 0:
        ppl = math.exp(loss)
        print(f"Epoch [{epoch:02d}/{CONFIG['num_epochs']}]  "
              f"Loss: {loss:.4f}  PPL: {ppl:7.2f}  LR: {lr_now:.2e}  ← Perplexity")
    else:
        print(f"Epoch [{epoch:02d}/{CONFIG['num_epochs']}]  "
              f"Loss: {loss:.4f}  LR: {lr_now:.2e}")

print("\n训练完成！")
print(f"最终 Loss: {epoch_losses[-1]:.4f}  "
      f"最终 PPL: {math.exp(epoch_losses[-1]):.2f}")

#### 5.2 保存模型与 Loss 曲线

In [ ]:
# 保存模型和词表，下次可以直接加载生成，不用重新训练
torch.save({
    "model_state_dict": model.state_dict(),
    "char2idx": char2idx,
    "idx2char":  idx2char,
    "vocab":     vocab,
    "config":    CONFIG,
}, CONFIG["save_model"])
print(f"模型已保存 → {CONFIG['save_model']}")

训练完成后，绘制 Loss 收敛曲线（图 1），观察损失随训练轮次的变化趋势。

In [ ]:
# 画 loss 曲线，看看收敛情况
fig, ax = plt.subplots(figsize=(9, 5))
epochs_x = list(range(1, len(epoch_losses) + 1))
ax.plot(epochs_x, epoch_losses, "g-o", markersize=2,
        linewidth=1.8, label="Train Loss")
ax.set_xlabel("Epoch", fontsize=13)
ax.set_ylabel("Loss",  fontsize=13)
ax.set_title("Training Loss Curve", fontsize=15)
ax.legend(fontsize=12)
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_xticks(epochs_x)
fig.tight_layout()
fig.savefig(CONFIG["loss_fig"], dpi=150)
plt.show()
print(f"Loss 曲线 → {CONFIG['loss_fig']}")

#### 5.3 训练过程分析

按学习率变化将 60 轮训练划分为四个阶段进行观察：

| 阶段 | Epoch | 学习率 | Loss 变化 | 每轮降幅 |
|------|-------|--------|----------|---------|
| 1 | 1–14 | 1.00×10⁻³ | 6.57 → 4.23 | 0.167 |
| 2 | 15–29 | 7.00×10⁻⁴ | 4.17 → 3.48 | 0.046 |
| 3 | 30–44 | 4.90×10⁻⁴ | 3.45 → 2.99 | 0.030 |
| 4 | 45–60 | 3.43×10⁻⁴ | 2.97 → 2.64 | 0.020 |

第一阶段学习速率最高（平均每轮降 0.167），模型在高学习率下快速捕捉字间基本搭配规律。后续阶段随学习率衰减，每轮降幅递减，但损失始终保持下降趋势。最后 5 轮单步降幅在 0.014～0.018 间波动，未见明显收敛平台，模型仍有继续训练的空间。

困惑度由第 5 轮的 146.31 降至第 60 轮的 14.06，模型将 6,564 类上的不确定性压缩了约两个数量级。需指出，该 PPL 仅在训练集上计算，参数/token 比为 36.3，泛化 PPL 预计将明显高于此值。

与初始配置（Dropout=0.1, StepLR step=10, γ=0.5）的对比：

| 指标 | 初始配置 | 本文配置 | 变化 |
|------|---------|--------|------|
| 最终 Loss | 3.89 | 2.64 | ↓32% |
| 最终 PPL | 48.8 | 14.06 | ↓71% |
| 末轮学习率 | 1.56×10⁻⁵ | 2.40×10⁻⁴ | ↑15× |
| 参数量 | 33.18M | 26.98M | ↓19% |
| 后 20 轮 Loss 降幅 | 0.145 | 0.442 | ↑3.0× |

改善主要来自两方面。一是更温和的学习率衰减使后期训练效率提升约 3 倍，避免了学习率过早衰减至无效区间。二是权重共享减少约 19% 参数量，在一定程度上缓解了过拟合。

### 6. 诗歌生成与质量评估

#### 6.1 生成策略

生成采用自回归采样。先将 `[START]` 与起始词「明月」送入模型做一次前向传播，使 LSTM 隐藏状态获得初始上下文；随后从「月」字开始，每一步将上一步输出的字符作为下一步输入，逐步生成后续字符。

为保证输出符合七言绝句格律，生成过程施加硬约束：在固定位置（第 7、15、23、31 位）直接插入对应标点（逗号或句号），其余位置屏蔽全部标点 token。无论采样结果如何，输出始终为标准的七言×四句格式。

温度参数设为 0.8。实验比较了 T=0.6、0.8、1.0 三个取值：T=0.6 时生成内容重复严重、多样性不足；T=1.0 时易出现不合理的字词组合；T=0.8 在多样性与连贯性间取得了可接受的结果。

In [ ]:
# 七言绝句的标点位置：第7位逗号，第15位句号，第23位逗号，第31位句号
PUNCT_MAP   = {7: "，", 15: "。", 23: "，", 31: "。"}
ALL_PUNCTS  = set("，。！？；、,.")


def generate(model, start_words, char2idx, idx2char, device,
             temperature=1.0, seq_len=32):
    """
    给定起始词（如"明月"），自回归生成一首七言绝句。
    返回 4 行字符串。
    """
    model.eval()
    unk_id   = char2idx[UNK]
    start_id = char2idx[START]
    all_punct_ids = [char2idx[p] for p in ALL_PUNCTS if p in char2idx]

    generated = list(start_words)

    with torch.no_grad():
        # 先用 [START] + start_words 预热，让 hidden state 有上下文
        primer = [start_id] + [char2idx.get(c, unk_id) for c in start_words]
        inp    = torch.tensor([primer], dtype=torch.long, device=device)
        hidden = model.init_hidden(1, device)
        _, hidden = model(inp, hidden)

        # 从 start_words 最后一个字开始，一步步往后生成
        last_id = char2idx.get(start_words[-1], unk_id)
        inp = torch.tensor([[last_id]], dtype=torch.long, device=device)

        while len(generated) < seq_len:
            logits, hidden = model(inp, hidden)
            logit = logits[0, 0].clone() / temperature

            pos = len(generated)

            if pos in PUNCT_MAP:
                # 标点位置直接插入，不用采样
                next_char = PUNCT_MAP[pos]
            else:
                # 非标点位置屏蔽标点 token，然后温度采样
                logit[all_punct_ids] = -1e9
                probs = torch.softmax(logit, dim=-1)
                next_id = torch.multinomial(probs, 1).item()
                next_char = idx2char.get(next_id, UNK)

            generated.append(next_char)
            inp = torch.tensor(
                [[char2idx.get(next_char, unk_id)]],
                dtype=torch.long, device=device
            )

    # 拆成 4 行
    s     = "".join(generated[:seq_len])
    lines = [s[i*8:(i+1)*8] for i in range(4)]
    return "\n".join(lines)

print("生成函数定义完毕。")

以下为以「明月」为起始词生成的 5 首七言绝句。

In [ ]:
# 生成 5 首诗看看效果
print("=" * 50)
print(f'以「{CONFIG["start_words"]}」为起始词生成七言绝句：')
print("=" * 50)

for i in range(5):
    poem = generate(
        model, CONFIG["start_words"],
        char2idx, idx2char, DEVICE,
        temperature=CONFIG["temperature"]
    )
    print(f"\n【第 {i+1} 首】\n{poem}")

print("\n" + "=" * 50)

#### 6.2 生成结果分析

对 5 首生成结果逐首讨论如下。

**第 1 首**：「明月明波影裏遊，此涼應不爲清秋。斜陽作陣湖山雨，洗盡塵埃數日愁。」整体意境较为完整：前两句写月夜水畔的凉意，第三句转入斜阳雨景，末句以「洗盡塵埃數日愁」收束，起承转合基本清晰。「斜陽作陣湖山雨」中「作陣」在宋诗中有类似用例。不足之处在于首句「明月明波」中「明」字紧邻重复，读来不够顺畅。

**第 2 首**：「明月明如火炬天，此時此夜轉孤燈。兵邊物物自無寐，獨倚蒲團夜夢中。」意象覆盖较广（火炬、孤灯、蒲团、夜梦），但「兵邊」与后文「蒲團」之间意境关联薄弱，像是将训练集中不同语境下的片段直接拼合。「此時此夜」与「物物」连续用叠字，稍显刻意。

**第 3 首**：「明月塘風雨夜凉，夢魂猶在鐵冠前。龍深日照長安近，山上龍門一榻清。」出现了「鐵冠」「長安」「龍門」等古典词汇，有基本的古风面貌。但「龍」字在同一首中出现两次（「龍深」「龍門」），且「明月塘」这一搭配在宋诗中少见，可能是模型基于局部 bigram 概率拼合的结果。

**第 4 首**：「明月明溪氣清輝，棹來風露不勝涼。如何一夜蟾松露，未減人間萬木花。」这首在 5 首中质量相对最好。意象从月光到溪水，延伸至风露、松露，最后落于花木，各句之间有一定递进关系。结句「未減人間萬木花」带有转折意味——露水虽凉而花木依旧，具备一定诗感。

**第 5 首**：「明月明行照初干，此月今宵月在天。夜靜釣魚天籟息，雲頭頻見一雙橫。」问题最为集中：「月」字在一首诗中出现 5 次（明月、此月、月在天），重复程度过高；「明行照初干」语义不通；「一雙橫」收尾也较为突兀。

**整体观察。** 格律层面，5 首结果均正确符合 7+7+7+7 格式，这是强制标点约束的直接效果。内容层面存在两个突出问题。

一是起始位置的字符重复。5 首中有 4 首以「明月明」开头——宋诗咏月题材占比大，「月→明」是训练集中最高频的 bigram 之一，模型学到这一搭配后过度外推，每次见到「明月」都倾向接「明」字。同首诗内也存在字符反复出现的问题（如第 5 首的「月」），模型缺乏对已生成字符的抑制机制。

二是跨句主题连贯性不足。单句层面大多通顺，但四句合成一首时常见主题跳跃。第 2 首从「火炬天」跳到「孤灯」再跳到「蒲团」，场景切换缺乏过渡。LSTM 对 32 个 token 跨度的长距离结构建模能力有限——像人类作者那样围绕统一主题谋篇布局，仅靠逐字递归生成仍有较大距离。

针对上述问题，可考虑的改进包括：（1）采样时引入重复惩罚（Repetition Penalty），对已出现 token 的概率施加衰减；（2）使用 Nucleus Sampling（Top-P）替代纯温度采样，截断低概率 token 的长尾。

### 7. 结论与讨论

#### 7.1 主要工作与发现

本文完成了一个从数据处理到模型训练再到生成评估的古诗自动生成完整流程。主要发现如下。

**数据方面。** 从 255 个宋诗 JSON 文件中筛选出 23,213 首七言绝句，构建了 |V|=6,564 的字符级词表。该数据规模对 27M 参数量级的模型而言偏小，参数/token 比约 36.3，过拟合是需要持续关注的问题。

**模型方面。** 3 层 LSTM 结合权重共享的架构在可用数据上取得了可接受的训练效果。权重共享在减少 19% 参数的同时，通过约束输入输出空间一致起到了一定的正则化作用。投影层的引入解决了 hidden size 与 embedding dim 不一致的问题。

**训练方面。** 学习率衰减策略的调整——γ 由 0.5 升至 0.7，step 由 10 升至 15——是收效最明显的单项改动。初始配置中学习率过早衰减至 10⁻⁵ 量级，后期约 20 轮训练几乎无进展；调整后第 60 轮学习率维持在 2.4×10⁻⁴，每轮损失仍有约 0.016 的降幅，尚未完全收敛。

**生成方面。** 模型能稳定输出格律正确的七言绝句，词汇以古典诗词常用字为主。但存在两个突出问题：起始位置 bigram 重复（根源于训练集分布偏斜），以及跨句主题统一性不足（受限于 LSTM 的长距离建模能力）。

#### 7.2 局限与后续方向

本文工作存在若干不足。未设置验证集，所有 PPL 指标仅在训练集上报告，缺乏对泛化性能的判断。生成策略仅采用了温度采样，未实现 Top-K 或 Nucleus Sampling。此外，重复惩罚机制的缺失直接导致了频繁的起始重复问题。

若继续推进该方向，以下几项值得优先尝试：（1）划分 10%～15% 数据作为验证集，监控 train/val 损失差距，必要时引入早停；（2）在采样过程中加入重复惩罚，降低已出现 token 被采样的概率；（3）将所用数据文件从 80 个扩展至全量 255 个以扩大训练规模；（4）在 LSTM 之上叠加 Self-Attention 层以增强跨句建模能力；（5）尝试基于 Transformer 的架构，对比不同序列模型在该任务上的表现差异。

### 参考文献

[1] Hochreiter S, Schmidhuber J. Long Short-Term Memory[J]. Neural Computation, 1997, 9(8): 1735-1780.

[2] Press O, Wolf L. Using the Output Embedding to Improve Language Models[C]. EACL, 2017.

[3] Zhang X, Lapata M. Chinese Poetry Generation with Recurrent Neural Networks[C]. EMNLP, 2014.

[4] Wang Z, et al. Chinese Song Iambics Generation with Neural Attention-Based Model[C]. IJCAI, 2016.

[5] Mikolov T, et al. Recurrent Neural Network based Language Model[C]. INTERSPEECH, 2010.

[6] Merity S, et al. Regularizing and Optimizing LSTM Language Models[C]. ICLR, 2018.